# Sistemas de Equações

Um [sistema de equações diferenciais](https://en.wikipedia.org/wiki/System_of_differential_equations) é um conjunto de equações envolvendo funções desconhecidas $u_0,\dots,u_{N-1}$ e suas derivadas. A dimensão de um sistema é o número $N$ de funções desconhecidas. A ordem do sistema é a derivada de maior ordem que aparece no conjunto de equações. Todo sistema de equações diferenciais é equivalente a um sistema de primeira ordem em uma dimensão superior.

In [ ]:
import numpy as np
import scipy.integrate as spi
import matplotlib.pyplot as plt

## Sistemas de Primeira Ordem

Todo sistema de equações diferenciais é equivalente a um sistema de primeira ordem em uma dimensão superior. Por exemplo, considere uma equação diferencial de segunda ordem

$$
ay'' + by' + cy = F(t)
$$

onde $a,b,c$ são constantes (com $a \ne 0$) e $F(t)$ é uma função conhecida. A função desconhecida $y(t)$ é de segunda ordem na equação, portanto, introduzimos duas novas variáveis $u_0 = y$ e $u_1 = y'$ e escrevemos a equação de segunda ordem 1-dimensional como um sistema de primeira ordem 2-dimensional

\begin{align}
u_0' &= u_1 \\\
u_1' &= (F(t) - b u_1 - c u_0)/a
\end{align}

O procedimento para qualquer sistema é semelhante:

1. Identifique a ordem de cada função desconhecida no sistema.
2. Se $y$ tem ordem $n$, então introduza $n$ novas variáveis $u_0 = y,u_1=y',\dots,u_{n-1}=y^{(n-1)}$.
3. Reescreva as equações apenas em termos das novas variáveis.

Por exemplo, considere o sistema

\begin{align}
x' &= x + y \\\
y'' &= xy' + y
\end{align}

Introduza novas variáveis $u_0 = x, u_1 = y, u_2 = y'$ e reescreva o sistema

\begin{align}
u_0' &= u_0 + u_1 \\\
u_1' &= u_2 \\\
u_2' &= u_0 u_2 + u_1
\end{align}

## Notação Vetorial

Um sistema de equações diferenciais de primeira ordem de dimensão $N$ é da forma

\begin{align}
u_0' &= f_0(t,u_0,\dots,u_{N-1}) \\\
u_1' &= f_1(t,u_0,\dots,u_{N-1}) \\\
& \ \ \vdots \\\
u_{N-1}' &= f_{N-1}(t,u_0,\dots,u_{N-1})
\end{align}

Escreva o sistema em notação vetorial

$$
\frac{d \mathbf{u}}{dt} = \mathbf{f}(t,\mathbf{u})
$$

onde

$$
\mathbf{u} = \begin{bmatrix} u_0 \\\ \vdots \\\ u_{N-1} \end{bmatrix}
\hspace{10mm}
\frac{d \mathbf{u}}{dt} = \begin{bmatrix} u_0' \\\ \vdots \\\ u_{N-1}' \end{bmatrix}
\hspace{10mm}
\mathbf{f}(t,\mathbf{u}) = \begin{bmatrix} f_0(t,u_0,\dots,u_{N-1}) \\\ \vdots \\\ f_{N-1}(t,u_0,\dots,u_{N-1}) \end{bmatrix}
$$

## Método de Euler

Como aplicamos o método de Euler a um sistema de equações de primeira ordem? Simplesmente aplicamos o método a cada função desconhecida no sistema.

### Equações de Segunda Ordem

Considere uma equação diferencial de segunda ordem com coeficientes constantes

$$
ay'' + by' + cy = F(t) \ , \ y(0)=y_0 \ , \ y'(0)=v_0
$$

Aplique o método de Euler a $y$ e $y'$ simultaneamente:

\begin{align}
y_{n+1} &= y_n + y_n' h \\\
y_{n+1}' &= y_n' + y_n'' h
\end{align}

onde

$$
y_n'' = (F(t_n) - cy_n - by_n')/a
$$

Por exemplo, considere a equação $y'' + y = 0$, $y(0)=1$, $y'(0)=0$. Sabemos que a solução exata é $y(t) = \cos(t)$. Calcule a aproximação pelo método de Euler e compare com a solução exata.

In [ ]:
y0 = 1; v0 = 0;
t = np.linspace(0,3*2*np.pi,3*100)
y = np.zeros(len(t))
y[0] = y0
dy = np.zeros(len(t))
dy[0] = v0
for n in range(0,len(t)-1):
    h = t[n+1] - t[n]
    y[n+1] = y[n] + dy[n]*h
    dy[n+1] = dy[n] - y[n]*h
plt.plot(t,y,'b',t,np.cos(t),'r'), plt.grid(True)
plt.title(r"$y'' + y = 0 , y(0) = 1 , y'(0) = 0$")
plt.legend(["Euler","Exata"])
plt.show()

Note que a amplitude da aproximação de Euler **cresce** a cada oscilação, enquanto a solução exata $\cos(t)$ tem amplitude constante. Uma forma de enxergar isso é pela energia $E = \frac{1}{2}\left(y^2 + (y')^2\right)$, que é conservada pela solução exata ($E' = yy' + y'y'' = yy' - y'y = 0$). No método de Euler,

$$
y_{n+1}^2 + (y'_{n+1})^2 = (y_n + y'_n h)^2 + (y'_n - y_n h)^2 = (1 + h^2)\left(y_n^2 + (y'_n)^2\right),
$$

ou seja, a energia numérica é multiplicada por $1 + h^2 > 1$ a cada passo. Diminuir $h$ reduz esse crescimento, mas não o elimina.

In [ ]:
E = (y**2 + dy**2)/2
plt.plot(t,E,'b',t,0.5 + 0*t,'r'), plt.grid(True)
plt.title("Energia da aproximação de Euler"), plt.xlabel("$t$"), plt.ylabel("$E$")
plt.legend(["Euler","Exata"])
plt.show()

### Implementação

Considere um sistema de primeira ordem em notação vetorial

$$
\frac{d \mathbf{u}}{dt} = \mathbf{f}(t,\mathbf{u})
$$

A fórmula para o método de Euler é quase exatamente a mesma que para equações escalares

$$
\mathbf{u}_{n+1} = \mathbf{u}_n + \mathbf{f}(t_n,\mathbf{u}_n) h
$$

onde $\mathbf{u}_n$ é o vetor de valores no passo $n$

$$
\begin{bmatrix} u_{0,n} \\\ \vdots \\\ u_{N-1,n} \end{bmatrix}
$$

tal que $u_{k,n} \approx u_k(t_n)$ no passo $n$ para $k=0,\dots,N-1$.

Escreva uma função chamada `odeEuler` que receba os parâmetros de entrada `f`, `t` e `u0` onde:

* `f` é uma função que representa o lado direito da equação $\mathbf{u}' = \mathbf{f}(t,\mathbf{u})$
* `t` é um array NumPy 1D
* `u0` é um valor inicial $\mathbf{u}(t_0)=\mathbf{u}_0$ onde $t_0$ é o valor `t[0]`

A função `odeEuler` retorna um array NumPy 2D `U` de tamanho `(len(t),len(u0))` com valores $u_k(t_n)$ na coluna $k$ e linha $n$.

A implementação abaixo é **a mesma** que usamos para equações escalares: como convertemos `y0` e os valores de `f` em arrays NumPy (`np.asarray`), a operação `y[n] + k1*h` funciona tanto para números quanto para vetores. Sem essa conversão, uma função `f` que retornasse uma **lista** causaria erro, pois `lista*h` não é uma operação vetorial em Python.

In [ ]:
def odeEuler(f,t,y0):
    """Método de Euler para y' = f(t,y), y(t[0]) = y0.
    Funciona para equações escalares (y0 número) e para sistemas (y0 lista/array)."""
    y0 = np.asarray(y0, dtype=float)
    y = np.zeros((len(t),) + y0.shape)
    y[0] = y0
    for n in range(0,len(t)-1):
        h = t[n+1] - t[n]
        k1 = np.asarray(f(t[n],y[n]))
        y[n+1] = y[n] + k1*h
    return y

Teste a função `odeEuler` no exemplo $y'' + y = 0, y(0)=1, y'(0)=0$ e compare com a solução exata $y(t) = \cos(t)$.

In [ ]:
def f(t,u):
    dudt = np.zeros(2)
    dudt[0] = u[1]
    dudt[1] = -u[0]
    return dudt

t = np.linspace(0,3*2*np.pi,3*100)
u0 = [1,0]
U = odeEuler(f,t,u0)
plt.plot(t,U[:,0],'b',t,np.cos(t),'r'), plt.grid(True)
plt.title(r"$y'' + y = 0 , y(0) = 1 , y'(0) = 0$")
plt.legend(["Euler","Exata"])
plt.show()

## scipy.integrate.odeint

A função `scipy.integrate.odeint` funciona da mesma forma que nossa função `odeEuler` acima, exceto que a ordem de $t$ e $\mathbf{u}$ é invertida

$$
\frac{d \mathbf{u}}{dt} = \mathbf{f}(\mathbf{u},t)
$$

Por exemplo, use `odeint` para aproximar a solução da equação de segunda ordem

$$
y'' + y = 0 \ , \ \ y(0)=1 \ , \ \ y'(0)=0
$$

e compare com a solução exata. O resultado é muito mais preciso que o método de Euler!

In [ ]:
def f(u,t):
    dudt = np.zeros(2)
    dudt[0] = u[1]
    dudt[1] = -u[0]
    return dudt

t = np.linspace(0,2*2*np.pi,2*50)
u0 = [1,0]
U = spi.odeint(f,u0,t)
plt.plot(t,U[:,0],'b.',t,np.cos(t),'r'), plt.grid(True)
plt.title(r"$y'' + y = 0 , y(0) = 1 , y'(0) = 0$")
plt.legend(["odeint","Exata"])
plt.show()

## Retrato de Fase e Campo Vetorial

Para um sistema autônomo bidimensional

\begin{align}
x' &= f(x,y) \\\
y' &= g(x,y)
\end{align}

o **plano de fase** é o plano $xy$. Cada solução $(x(t),y(t))$ descreve uma curva nesse plano (uma **trajetória** ou **órbita**), e o vetor $(f(x,y),g(x,y))$ é tangente à trajetória em cada ponto. O conjunto desses vetores é o **campo vetorial** do sistema, análogo bidimensional do campo de direções. O **retrato de fase** é o desenho do campo vetorial junto com algumas trajetórias.

No Matplotlib, o campo vetorial pode ser desenhado com `plt.quiver` (setas) ou com `plt.streamplot` (linhas de fluxo). Ambas recebem malhas `X, Y` criadas com `np.meshgrid` e os valores das componentes `U, V` do campo nessa malha.

Por exemplo, a equação do oscilador amortecido $y'' + 0.3y' + y = 0$ corresponde ao sistema $x' = y$, $y' = -x - 0.3y$ (com $x$ a posição e $y$ a velocidade).

In [ ]:
def f(u,t):
    return [u[1], -u[0] - 0.3*u[1]]

# campo vetorial em uma malha
x = np.linspace(-3,3,25); y = np.linspace(-3,3,25)
X, Y = np.meshgrid(x,y)
U = Y                     # x' = y
V = -X - 0.3*Y            # y' = -x - 0.3y
plt.figure(figsize=(6,6))
plt.streamplot(X,Y,U,V,color='lightgray',density=1.2)

# algumas trajetórias
t = np.linspace(0,30,1000)
for u0 in [[2.5,0],[0,2.5],[-2,-2]]:
    sol = spi.odeint(f,u0,t)
    plt.plot(sol[:,0],sol[:,1],lw=2)
    plt.plot(u0[0],u0[1],'ko')

plt.plot(0,0,'r*',markersize=12)      # equilíbrio
plt.xlabel("posição $x$"), plt.ylabel("velocidade $y$")
plt.title(r"Retrato de fase de $y'' + 0.3y' + y = 0$")
plt.axis('square'), plt.xlim([-3,3]), plt.ylim([-3,3])
plt.show()

As trajetórias espiralam em direção ao equilíbrio $(0,0)$: o oscilador perde energia e, a cada volta, a amplitude e a velocidade máximas diminuem. Experimente trocar o coeficiente $0.3$ por $0$ (sem amortecimento) ou por $3$ (amortecimento forte) e observe como o retrato de fase muda.